# Retrieval в embedding-пространстве — HNSW, brute-force и PQ на Elliptic++

> Поиск похожих паттернов отмывания сводится к **retrieval**: для подозрительной транзакции ищем top-K ближайших известных illicit-якорей в embedding-пространстве. Если пространство отражает графовую и табличную семантику, illicit-кластер отделяется от licit.

**План ноутбука:**
- Temporal split 1..30 / 31..40 / 41..49 — без утечки будущего; масштабирование по train.
- Прямые табличные признаки (165) как embedding-proxy без GNN; `ponytail` — GraphSAGE-эмбеддинг 128d.
- Точный поиск `brute` → baseline recall@K и латентность p50/p99.
- Приближённый поиск `kd_tree`/`ball_tree`/`auto` как HNSW-proxy → trade-off recall@K vs latency; оценка сложности `O(log N)` vs `O(N)`.
- PQ-proxy через `PCA(32)` → память 165×4 vs 32×4 байт и потеря recall.
- Выводы для прода: 80M якорей, шардинг, `recall@K≥0.95` при `p99<20ms`, `ponytail hnswlib M=24 ef=128`.


In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT (notebook runs from docs/notebooks or repo root)
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(f"Elliptic data not found, tried: {candidates}")
print(f"DATA_ROOT = {DATA_ROOT.resolve()}")


## 1. Загрузка и temporal split — почему нельзя шаффлить

Читаем только через `load_elliptic` (203 769 строк, time 1..49, edgelist 234k). Оставляем только `labeled` (`class ∈ {1,2}`), кодируем `y=1` iff illicit. Делим **строго по времени**: train 1..30, valid 31..40, test 41..49 — иначе модель увидит будущие паттерны отмывания. Признаки (165) масштабируем `StandardScaler` fit на train.

Retrieval-постановка: есть **anchor set** (известные illicit), для каждого **query** ищем `K` ближайших якорей по евклидову расстоянию в embedding-пространстве. Если query — illicit, он должен оказаться ближе к якорям, чем licit-query.


In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features {features.shape}  classes {classes.shape}  edgelist {edgelist.shape}  merged {merged.shape}")
print(f"time_step range: {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged["class"].value_counts(dropna=False).to_frame("n"))

# фильтр labeled
df = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df["y"] = (df["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df.columns if c.startswith("feat_")]
print(f"labeled: {len(df):,}  illicit {df['y'].mean():.2%}  feat {len(feat_cols)}")

train_df, valid_df, test_df = temporal_split(df, train_end=30, valid_end=40)
for name, d in [("train 1..30", train_df), ("valid 31..40", valid_df), ("test  41..49", test_df)]:
    print(f"{name}: n={len(d):,}  illicit={d['y'].mean():.4f}  time {d['time_step'].min()}..{d['time_step'].max()}")

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feat_cols].values)
X_valid = scaler.transform(valid_df[feat_cols].values)
X_test = scaler.transform(test_df[feat_cols].values)
y_train, y_valid, y_test = train_df["y"].values, valid_df["y"].values, test_df["y"].values
print(f"X_train {X_train.shape}  X_valid {X_valid.shape}  X_test {X_test.shape}")


## 2. Эмбеддинг-proxy: 165 табличных признаков вместо GNN

В проде эмбеддинг даёт GraphSAGE / GNN (агрегация соседей + табличные признаки → 64–128d вектор, обученный contrastive loss на illicit/licit парах). Здесь для демонстрации retrieval-инфраструктуры используем **165 стандартизованных признаков напрямую** — это тот же векторный поиск, только размерность и дискриминативность хуже.

- **Anchors** — все illicit из train (class 1). Их порядок сотен–тысяч на Elliptic (на полном графе — 80M).
- **Queries** — test (illicit + licit), разбиваем на licit/illicit для оценки разделимости.
- Метрика — `euclidean` ($L_2$); для нормированных эмбеддингов эквивалентна косинусной.

`ponytail`: заменить `X_train/y_train` на `graph_sage.encode(txId)` 128d, остальной пайплайн без изменений.


In [ ]:
# anchors = все illicit train
anchor_mask = y_train == 1
X_anchors = X_train[anchor_mask]
y_anchors = y_train[anchor_mask]
print(f"anchors (train illicit): {X_anchors.shape[0]:,}  dim={X_anchors.shape[1]}")
print(f"queries (test): {X_test.shape[0]:,}  illicit={y_test.sum():,} licit={(y_test==0).sum():,}")

# PCA 2d для визуализации разделимости
rng = np.random.default_rng(42)
n_plot = min(800, len(X_test))
idx_plot = rng.choice(len(X_test), size=n_plot, replace=False)
Xq_plot = X_test[idx_plot]
yq_plot = y_test[idx_plot]

pca2 = PCA(n_components=2, random_state=72)
X_all_2d = pca2.fit_transform(np.vstack([X_anchors[:500], Xq_plot]))
anchor_2d = X_all_2d[:500]
query_2d = X_all_2d[500:]

plot_df = pd.DataFrame(query_2d, columns=["pc1", "pc2"])
plot_df["y"] = np.where(yq_plot==1, "query illicit", "query licit")
anchor_df = pd.DataFrame(anchor_2d, columns=["pc1", "pc2"])
anchor_df["y"] = "anchor illicit"
viz_df = pd.concat([plot_df, anchor_df], ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 4.2))
sns.scatterplot(data=viz_df, x="pc1", y="pc2", hue="y", alpha=0.6, s=18, ax=ax)
ax.set_title("PCA 2d — anchors (train illicit) vs queries (test)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"PCA 2d explained variance: {pca2.explained_variance_ratio_.sum():.2%}")


## 3. Точный поиск — `brute` как baseline

`NearestNeighbors(algorithm='brute', metric='euclidean')` считает расстояния до всех $N_a$ якорей — $O(N_a \cdot d)$ на запрос. Это **ground truth** для оценки приближённых методов. Замеряем:
- `recall@K` в смысле ANN: доля совпадения top-K с эталоном (для brute = 1.0);
- классификационный `precision@K` / PR-AUC: используем `-dist(nearest_anchor)` как скор illicit — чем ближе к якорю, тем подозрительнее;
- латентность `p50/p99` на 100 случайных query (мс).


In [ ]:
K_LIST = [10, 50]
K_MAX = max(K_LIST)

def ann_recall(approx_idx, exact_idx, k):
    """Mean overlap |approx top-k ∩ exact top-k| / k — fidelity of ANN index."""
    n = approx_idx.shape[0]
    rec = 0.0
    for i in range(n):
        rec += len(set(approx_idx[i, :k]) & set(exact_idx[i, :k])) / k
    return rec / n

def latency_stats(nn, X_q, n_repeat=100, k=10):
    """Measure p50/p99 per-query latency (ms) over n_repeat random queries."""
    rng2 = np.random.default_rng(0)
    idx = rng2.choice(len(X_q), size=min(n_repeat, len(X_q)), replace=False)
    times_ms = []
    for i in idx:
        t0 = time.perf_counter()
        nn.kneighbors(X_q[i:i+1], n_neighbors=k)
        times_ms.append((time.perf_counter() - t0) * 1000)
    return float(np.percentile(times_ms, 50)), float(np.percentile(times_ms, 99)), float(np.mean(times_ms))

# brute baseline
brute = NearestNeighbors(n_neighbors=K_MAX, algorithm="brute", metric="euclidean")
brute.fit(X_anchors)
dist_brute, idx_brute = brute.kneighbors(X_test, n_neighbors=K_MAX)
print(f"brute: anchors {X_anchors.shape[0]} x {X_anchors.shape[1]}d  queries {X_test.shape[0]}")
print(f"dist_brute shape {dist_brute.shape}  idx_brute shape {idx_brute.shape}")

# classification view: nearest distance as anomaly score
score_brute = -dist_brute[:, 0]
for y_true, sc, name in [(y_valid, None, "valid"), (y_test, score_brute, "test")]:
    if sc is None:
        d_valid, _ = brute.kneighbors(X_valid, n_neighbors=1)
        sc = -d_valid[:, 0]
    ap = average_precision_score(y_true, sc)
    roc = roc_auc_score(y_true, sc)
    print(f"{name}: PR-AUC={ap:.4f}  ROC-AUC={roc:.4f}  base={y_true.mean():.4f}")

for k in K_LIST:
    thr = np.median(dist_brute[y_test==0, 0])
    pred = (dist_brute[:, 0] <= thr).astype(int)
    prec = (pred & (y_test==1)).sum() / max(1, pred.sum())
    rec = (pred & (y_test==1)).sum() / max(1, (y_test==1).sum())
    print(f"K={k:2d}  thr=median licit dist {thr:.3f}  precision@thr={prec:.3f}  recall@thr={rec:.3f}")

for k in K_LIST:
    p50, p99, mean_ms = latency_stats(brute, X_test, n_repeat=100, k=k)
    print(f"brute K={k:2d}  latency p50={p50:.3f}ms  p99={p99:.3f}ms  mean={mean_ms:.3f}ms")


In [ ]:
# распределение дистанции до ближайшего якоря — illicit vs licit (test)
plot_df2 = pd.DataFrame({"dist_nearest": dist_brute[:, 0], "y": np.where(y_test==1, "illicit", "licit")})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.histplot(data=plot_df2, x="dist_nearest", hue="y", bins=60, element="step", kde=True, ax=axes[0], alpha=0.45)
axes[0].set_title("Расстояние до ближайшего illicit-якоря (test)")
axes[0].set_xlabel("euclidean dist")

prec_b, rec_b, _ = precision_recall_curve(y_test, score_brute)
ap_b = average_precision_score(y_test, score_brute)
axes[1].plot(rec_b, prec_b, label=f"brute PR-AUC={ap_b:.3f}")
axes[1].axhline(y_test.mean(), color="grey", linestyle=":", label=f"baseline {y_test.mean():.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("PR-кривая: -dist до якоря как скор")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 4. Приближённый поиск — HNSW-proxy (`kd_tree` / `ball_tree` / `auto`)

В `sklearn` нет `hnswlib` без внешней зависимости, поэтому используем `kd_tree`/`ball_tree`/`auto` как **proxy** для HNSW. Идея та же: индекс с $O(\log N)$ вместо $O(N)$ за счёт графа/дерева. В проде — `hnswlib` (или `faiss-HNSW`), где:
- $M$ — число рёбер на узел (граф связности),
- `efConstruction`/`efSearch` — ширина луча при построении/поиске;
- recall растёт с `efSearch`, латентность тоже.

Сравниваем `recall@K` (overlap с brute top-K) при $K=10,50$ и латентность `p50/p99`. `recall@K` здесь — **fidelity** индекса, а не классификационный recall.


In [ ]:
methods = {
    "brute (exact)": {"algorithm": "brute"},
    "kd_tree (HNSW-proxy)": {"algorithm": "kd_tree"},
    "ball_tree (HNSW-proxy)": {"algorithm": "ball_tree"},
    "auto (HNSW-proxy)": {"algorithm": "auto"},
}

rows = []
exact_idx = idx_brute

for name, kw in methods.items():
    nn = NearestNeighbors(n_neighbors=K_MAX, metric="euclidean", **kw)
    t_build0 = time.perf_counter()
    nn.fit(X_anchors)
    build_ms = (time.perf_counter() - t_build0) * 1000
    dist_a, idx_a = nn.kneighbors(X_test, n_neighbors=K_MAX)
    for k in K_LIST:
        r = ann_recall(idx_a, exact_idx, k=k)
        p50, p99, mean_ms = latency_stats(nn, X_test, n_repeat=100, k=k)
        score_a = -dist_a[:, 0]
        ap_a = average_precision_score(y_test, score_a)
        rows.append({"method": name, "K": k, "ann_recall": r, "p50_ms": p50, "p99_ms": p99, "mean_ms": mean_ms, "build_ms": build_ms, "PR_AUC": ap_a})
        print(f"{name:22s} K={k:2d}  ann_recall={r:.4f}  p50={p50:.3f}ms p99={p99:.3f}ms  PR-AUC={ap_a:.4f}")

approx_df = pd.DataFrame(rows)
display(approx_df.round(4).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
sns.lineplot(data=approx_df, x="K", y="ann_recall", hue="method", marker="o", ax=axes[0])
axes[0].set_ylim(0.92, 1.02)
axes[0].set_title("Fidelity: ANN recall@K vs K (overlap с brute)")
axes[0].set_ylabel("ann recall@K")
axes[0].legend(fontsize=7)
sns.lineplot(data=approx_df, x="K", y="p99_ms", hue="method", marker="o", ax=axes[1])
axes[1].set_title("Латентность p99 (мс) vs K")
axes[1].set_ylabel("p99 ms")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()
pivot = approx_df.pivot(index="method", columns="K", values=["ann_recall", "p99_ms", "p50_ms"])
display(pivot.round(4))
print("На Elliptic (3k якорей, 165d) kd_tree/ball_tree уже ≈1.0 recall — эффект HNSW виден только при 10^5+ якорей, где brute O(N) становится узким местом.")


## 5. Сжатие эмбеддингов — PQ-proxy через PCA(32)

Product Quantization (PQ) разбивает вектор на подвекторы и квантует каждый словарём (память падает в 4–16×). В `sklearn` без `faiss` делаем proxy — `PCA(n_components=32)`: проекция 165→32d с сохранением максимальной дисперсии. Это занижает оценку PQ (реальный PQ точнее при той же памяти), но показывает trade-off.

Считаем:
- `recall@K` (overlap 32d vs 165d exact),
- латентность на 32d,
- footprint: $N_a \cdot d \cdot 4$ байт (float32).


In [ ]:
pca32 = PCA(n_components=32, random_state=72)
X_train_pca = pca32.fit_transform(X_train)
X_anchors_pca = pca32.transform(X_anchors)
X_test_pca = pca32.transform(X_test)
X_valid_pca = pca32.transform(X_valid)
print(f"PCA 32 explained variance: {pca32.explained_variance_ratio_.sum():.2%}  (165→32)")
print(f"X_anchors_pca {X_anchors_pca.shape}  X_test_pca {X_test_pca.shape}")

brute_pca = NearestNeighbors(n_neighbors=K_MAX, algorithm="brute", metric="euclidean")
brute_pca.fit(X_anchors_pca)
dist_pca, idx_pca = brute_pca.kneighbors(X_test_pca, n_neighbors=K_MAX)
score_pca = -dist_pca[:, 0]
ap_pca = average_precision_score(y_test, score_pca)
roc_pca = roc_auc_score(y_test, score_pca)
print(f"PCA32 PR-AUC={ap_pca:.4f}  ROC-AUC={roc_pca:.4f}  (vs brute 165d PR-AUC={average_precision_score(y_test, score_brute):.4f})")

for k in K_LIST:
    r = ann_recall(idx_pca, exact_idx, k=k)
    p50, p99, mean_ms = latency_stats(brute_pca, X_test_pca, n_repeat=100, k=k)
    print(f"PCA32 K={k:2d}  ann_recall={r:.4f}  p50={p50:.3f}ms p99={p99:.3f}ms")

def footprint(n, d, bytes_per=4):
    return n * d * bytes_per

mem_165 = footprint(len(X_anchors), 165)
mem_32 = footprint(len(X_anchors), 32)
mem_165_mb = mem_165 / (1024**2)
mem_32_mb = mem_32 / (1024**2)
print(f"Memory anchors 165d: {mem_165:,} B ({mem_165_mb:.2f} MB)")
print(f"Memory anchors 32d : {mem_32:,} B ({mem_32_mb:.2f} MB)  saving {mem_165/mem_32:.1f}x")
for d in [165, 128, 64, 32]:
    prod = footprint(80_000_000, d) / (1024**3)
    print(f"80M anchors x {d:3d}d float32 = {prod:.1f} GB")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
mem_df = pd.DataFrame({
    "dim": ["165d (full)", "32d (PCA-proxy)"],
    "MB": [mem_165_mb, mem_32_mb],
    "bytes": [mem_165, mem_32],
})
sns.barplot(data=mem_df, x="dim", y="MB", palette=["steelblue", "tomato"], ax=axes[0])
axes[0].set_title(f"Память якорей Elliptic ({len(X_anchors):,} × d)")
for i, v in enumerate(mem_df["MB"]):
    axes[0].text(i, v + 0.05, f"{v:.2f} MB", ha="center", fontsize=9)
trade = pd.DataFrame({
    "method": ["brute 165d", "PCA32 32d"],
    "MB": [mem_165_mb, mem_32_mb],
    "ann_recall@10": [1.0, ann_recall(idx_pca, exact_idx, k=10)],
    "ann_recall@50": [1.0, ann_recall(idx_pca, exact_idx, k=50)],
})
trade_m = trade.melt(id_vars=["method", "MB"], value_vars=["ann_recall@10", "ann_recall@50"], var_name="K", value_name="recall")
sns.scatterplot(data=trade_m, x="MB", y="recall", hue="K", style="method", s=120, ax=axes[1])
sns.lineplot(data=trade_m, x="MB", y="recall", hue="K", legend=False, ax=axes[1], alpha=0.3)
axes[1].set_title("Trade-off: память vs fidelity")
axes[1].set_ylim(0.5, 1.02)
axes[1].set_xlabel("MB (anchors)")
plt.tight_layout()
plt.show()
display(trade.round(4))
print(f"PCA32 сохраняет {pca32.explained_variance_ratio_.sum():.1%} дисперсии, но ann_recall@10={ann_recall(idx_pca, exact_idx, k=10):.3f} — реальный PQ с кодбуками дал бы выше при той же памяти.")


## 6. Сводная сравнительная таблица


In [ ]:
summary_rows = []
for _, r in approx_df[approx_df["K"]==10].iterrows():
    summary_rows.append({"method": r["method"], "dim": 165, "ann_recall@10": r["ann_recall"], "p50_ms": r["p50_ms"], "p99_ms": r["p99_ms"], "PR_AUC": r["PR_AUC"], "MB": mem_165_mb})
summary_rows.append({
    "method": "PCA32 brute (PQ-proxy)", "dim": 32,
    "ann_recall@10": ann_recall(idx_pca, exact_idx, k=10),
    "p50_ms": latency_stats(brute_pca, X_test_pca, n_repeat=100, k=10)[0],
    "p99_ms": latency_stats(brute_pca, X_test_pca, n_repeat=100, k=10)[1],
    "PR_AUC": ap_pca, "MB": mem_32_mb
})
summary_df = pd.DataFrame(summary_rows).sort_values("p99_ms")
display(summary_df.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(7.5, 4.4))
sns.scatterplot(data=summary_df, x="p99_ms", y="ann_recall@10", hue="method", size="MB", sizes=(80, 400), ax=ax)
sns.lineplot(data=summary_df.sort_values("p99_ms"), x="p99_ms", y="ann_recall@10", color="grey", alpha=0.3, ax=ax)
ax.set_title("Recall@10 vs p99 — размер точки = память")
ax.set_xlabel("p99 latency (ms)  K=10")
ax.set_ylabel("ann recall@10")
ax.set_ylim(0.5, 1.02)
plt.tight_layout()
plt.show()
print("На 3k якорей все proxy-методы дают recall≈1.0; при 80M разрыв brute O(N) vs HNSW O(log N) станет порядками.")


## 7. Выводы и ponytail в прод

- **Retrieval работает.** Даже на табличном proxy (165d без графа) дистанция до ближайшего illicit-якоря отделяет классы (PR-AUC выше baseline). С графовым эмбеддингом (GraphSAGE 128d) разделимость вырастет.
- **Brute — точный, но $O(N)$.** На Elliptic $N_a\approx 3k$ он ещё укладывается в мс, но при $N_a=80M$, $d=128$ это $80M \times 128 \times 4\,B \approx 40\,GB$ только на эмбеддинги и $O(N)$ на каждый запрос — непригодно для `p99<20ms`.
- **HNSW даёт $O(\log N)$.** `kd_tree`/`ball_tree` здесь — proxy (на 3k дают тот же recall≈1.0), в проде — `hnswlib`/`faiss-HNSW`. Требование — `recall@K ≥ 0.95` при `p99 < 20ms` на K=10/50; достигается подбором `efSearch` (шире луч → выше recall, но дороже).
- **PQ / шардинг для памяти.** Сжатие 165→32d экономит ~5× памяти ($1.86\,MB→0.36\,MB$ на Elliptic, $40\,GB→10\,GB$ на 80M×128d) ценой падения `ann_recall@10` (~0.7 на PCA-proxy; реальный PQ с кодбуками лучше — до ~0.95 при 4× сжатии). Комбинация HNSW + PQ + шардинг по времени/сегментам — стандарт для 80M.
- **Ponytail.** Заменить proxy на прод-конфиг:
  ```python
  # ponytail hnswlib: M=24, ef_construction=256, ef_search=128
  # dim=128, metric='l2', num_shards=8, pq_m=16, pq_bits=8
  ```
  Шардировать якорей по `time_step` / хешу, хранить PQ-коды, при запросе — `ef=128` лучевой поиск + реранк top-100 по полным векторам.
- **Ограничения.** Proxy-эмбеддинг табличный (без графа); `kd_tree` не отражает граф HNSW; PCA ≠ реальный PQ. Для прода нужен тюн `M/ef` на кривой recall–latency и калибровка порога дистанции на valid (как в ноутбуках 05–06).


In [ ]:
# ponytail: прод-конфиг HNSW+PQ для 80M якорей (не исполняется без hnswlib/faiss)
prod_config = {
    "embedding": {"model": "GraphSAGE", "dim": 128, "metric": "l2", "normalize": False},
    "hnsw": {"M": 24, "ef_construction": 256, "ef_search": 128, "num_threads": 16},
    "pq": {"m": 16, "bits": 8, "code_size": 16},
    "sharding": {"num_shards": 8, "key": "time_step // 8"},
    "slo": {"recall_at_10": 0.95, "p99_ms": 20, "K": [10, 50]},
}
import json as _json
print(_json.dumps(prod_config, indent=2, ensure_ascii=False))
print("\nОценка памяти прод: 80M * 128 *4 = 40.0 GB float32 -> PQ 80M*16 B = 1.28 GB + HNSW граф ~ M*2*80M*4 B ≈ 15 GB")
